# Strict fine-cell balance through an equilibrated RT0 flux

This tutorial declares the local and global primal equations first, then reconstructs a physical Darcy flux. The reconstruction minimizes its distance from the raw pressure gradient subject to the existing skeleton flux and **every fine-cell source balance**. This RT0 constrained minimizer is a different operator from the canonical RT2 moment reconstruction; the convergence theorem for that operator is not assigned to this one.

We use the smooth analytical pressure and identity permeability from the first numerical experiment in Barrenechea, Martins, Pereira and Valentin, *Multiscale Modeling & Simulation* 24 (2026), [doi:10.1137/24M1673073](https://doi.org/10.1137/24M1673073). The RT0 choice and constraint operator here define a separate analytical control, rather than a reproduction of its RT2 table.

$$
p=\sin(2\pi x)\sin(2\pi y),\qquad
\boldsymbol q=-\nabla p,\qquad
f=8\pi^2p,\qquad p|_{\partial\Omega}=0.
$$

The independent pressure and vector-flux errors use quadrature orders 12 and 16. Conforming global P3 baseline meshes with 16, 32 and 64 subdivisions are evaluated in the companion [canonical reconstruction tutorial](reconstruction_and_indicators.ipynb), with the same operator, data and norms.

In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Preserve this exact notebook as the numerical source checked by its records.
NOTEBOOK_RELATIVE = Path("notebooks/darcy/equilibrated_flux_workflow.ipynb")
source_declaration = os.environ.get("PYMHM_NOTEBOOK_SOURCE")
source_candidates = (
    (Path(source_declaration),) if source_declaration else
    (Path.cwd() / NOTEBOOK_RELATIVE.name, Path.cwd() / NOTEBOOK_RELATIVE)
)
NOTEBOOK_SOURCE = next((path for path in source_candidates if path.is_file()), None)
if NOTEBOOK_SOURCE is None:
    raise FileNotFoundError(
        "Keep this notebook in the launch directory, or set PYMHM_NOTEBOOK_SOURCE "
        "to its actual file before starting the kernel."
    )
notebook_source_bytes = NOTEBOOK_SOURCE.read_bytes()

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/a5fea51829343bf1daa4a823424857bbb20b373fd3b70be7682b8ff8ab63ae53/equilibrated_flux_workflow-companion.zip"
COMPANION_SHA256 = "a5fea51829343bf1daa4a823424857bbb20b373fd3b70be7682b8ff8ab63ae53"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/equilibrated_flux_workflow.ipynb", directory=ROOT)
# Stage actual source bytes, including their independent provenance fingerprint.
workspace_notebook = ROOT / NOTEBOOK_RELATIVE
workspace_notebook.parent.mkdir(parents=True, exist_ok=True)
workspace_notebook.write_bytes(notebook_source_bytes)


## 1. Declare the analytical data

The source is the analytical negative Laplacian, independent of the assembled matrix. Arrays below supply the same functions to error integration; UFL supplies the load in each local variational form.

In [ ]:
import numpy as np
import ufl
from threadpoolctl import threadpool_limits
from pymhm import (
    Equation, FaceSpace, LocalContext, MeshHierarchy, SkeletonSpace, TriangleMesh,
    assemble, bind_interface, bind_problem, columns, solve,
)
from pymhm.postprocessing.solutions import DarcySolution
from examples.tutorial_equilibrated_flux import equilibrated_diagnostics, equilibrated_display_samples
from examples.tutorial_recovery_acquisition import (
    read_recovery_record, recovery_provenance, write_recovery_record,
)
from examples.tutorial_recovery_plots import plot_recovery_fields
from examples.tutorial_convergence import refinement_series, plot_method_series, asymptotic_summary


def exact_pressure(points: np.ndarray) -> np.ndarray:
    """Boundary-compatible analytical pressure."""
    return np.sin(2*np.pi*points[:, 0])*np.sin(2*np.pi*points[:, 1])


def exact_gradient(points: np.ndarray) -> np.ndarray:
    """Independently differentiated analytical gradient."""
    x, y = 2*np.pi*points.T
    return 2*np.pi*np.column_stack((np.cos(x)*np.sin(y), np.sin(x)*np.cos(y)))


def source(points: np.ndarray) -> np.ndarray:
    """Independent negative Laplacian of the analytical pressure."""
    return 8*np.pi**2*exact_pressure(points)

RUN_STUDY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"
RECORD = ROOT/"examples/results/tutorial-methods/equilibrated-rt0-current.json"

## 2. Translate the variational equations directly into UFL

Each macrocell contains four fine triangles and a continuous local P3 pressure. A constant is retained through its physical integral. The multiplier represents the **outward physical normal Darcy flux** after PyMHM applies the declared normal-orientation convention. With one P0 segment per macroface, it restricts to constants on the aligned fine boundary edges.

$$
\begin{aligned}
(\nabla p_T,\nabla v)_T+\langle\lambda_T,v\rangle_{\partial T}
 &= (f,v)_T,\\
\sum_T\langle\mu_T,p_T\rangle_{\partial T}&=0
\quad\text{for homogeneous Dirichlet data.}
\end{aligned}
$$

The `b` pairing contributes the positive normal-flux term to the local equation. The opposite `c` pairing supplies the global pressure-continuity equation. The declared constant kernel and volume moment fix the local mean; arbitrary algebraic nullspace normalization is not a physical gauge.

In [ ]:
def local_equations(local: LocalContext):
    """Declare diffusion, source, trace pairings and the physical mean."""
    pressure = local.native_space(degree=3)
    p, v = ufl.TrialFunction(pressure.space), ufl.TestFunction(pressure.space)
    x = ufl.SpatialCoordinate(pressure.mesh)
    dx = ufl.Measure("dx", domain=pressure.mesh, metadata={"quadrature_degree": 20})
    f = 8*np.pi**2*ufl.sin(2*np.pi*x[0])*ufl.sin(2*np.pi*x[1])
    local.field("pressure", pressure)
    return local.equations(
        a=ufl.inner(ufl.grad(p), ufl.grad(v))*dx,
        L=f*v*dx,
        b=local.trace_pairings(lambda phi, ds: phi*v*ds),
        c=local.trace_pairings(lambda phi, ds: -phi*p*ds, axis="rows"),
        kernel=np.ones((pressure.size, 1)), moments=columns(v*dx),
    )


def global_equation(global_context):
    """Apply homogeneous weak Dirichlet data in the bound interface layout."""
    boundary, fixed = global_context.boundary_data(0.0, order=8)
    assert not fixed
    return Equation(0, global_context.trace_load(-boundary))

## 3. Choose meshes, assemble, solve and obtain the physical fields

The following function repeats only the workflow on a caller-selected macro mesh. The local equations above still define the operator. `DarcySolution` interprets the solved coefficients and does not solve another PDE. Its raw centroid flux samples are recorded together with the actual continuous P3 coefficient fields; integration evaluates full gradients at quadrature points.

The residual label below refers to original **global compatibility**. A separate native eight-subdivision observer verifies every original local volume row and the original trace equation for these same P3/P0 forms: maximum relative local defect 1.38e-14 and full uncondensed defect relative to the physical load 1.92e-14, against the unchanged 1e-10 gate. These algebraic diagnostics are distinct from physical vector errors and the strict fine-cell balances imposed afterward.


In [ ]:
def solve_declared(macro: TriangleMesh) -> DarcySolution:
    """Run the declared workflow with aligned P0 trace and P3 local pressure."""
    meshes = tuple(macro.submesh(cell, 2) for cell in range(len(macro.cells)))
    trace = SkeletonSpace(macro, tuple(FaceSpace.uniform(0) for _ in macro.faces))
    hierarchy = MeshHierarchy(macro, meshes)
    interface = bind_interface(trace, convention="normal")
    problem = bind_problem(hierarchy, interface, local_equations,
                           global_equation=global_equation, retained=1)
    coefficients = solve(assemble(problem))
    fields = coefficients.field("pressure")
    raw_flux = tuple(
        -field.gradient(mesh.points[mesh.cells].mean(axis=1), cells=np.arange(len(mesh.cells)))
        for mesh, field in zip(meshes, fields, strict=True)
    )
    return DarcySolution(
        skeleton=trace, local_meshes=meshes,
        pressure=tuple(field.portable_coefficients for field in fields), flux=raw_flux,
        hybrid=coefficients, formulation="primal", permeability=1.0,
        source=source, degree=3, quadrature_order=16,
    )


with threadpool_limits(1):
    state = solve_declared(TriangleMesh.unit_square(8))
print("Original equation residual:", state.hybrid.raw_residual)

## 4. Impose every fine-cell balance in RT0

On each macrocell, solve the existing constrained reconstruction:

$$
\begin{aligned}
\boldsymbol q_T^{\mathrm{eq}}
&=\underset{\boldsymbol r\in RT_0(\mathcal T_h^T)}{\operatorname{argmin}}
\|\boldsymbol r+\nabla p_T\|_{L^2(T)}^2,\\
\int_t\nabla\cdot\boldsymbol r&=\int_t f
\quad\text{for every fine cell }t,\\
\int_e\boldsymbol r\cdot\boldsymbol n_T&=\int_e\lambda_T
\quad\text{for every fine boundary edge }e.
\end{aligned}
$$

The skeleton conservation identity supplies compatibility between the source and boundary constraints. RT0 edge integrals use the same outward orientation as the solved trace. The algorithm checks this compatibility before returning a flux. Its divergence is the cell average of the source, rather than the pointwise analytical source.

In [ ]:
from pymhm.recovery.equilibrated import equilibrate_flux

with threadpool_limits(1):
    reconstructed = equilibrate_flux(state)
    diagnostics, reconstructed = equilibrated_diagnostics(state, exact_pressure, exact_gradient)
print("Maximum fine-cell integrated balance defect:", diagnostics["fine_cell_balance_defect"])
print("Physical RT0 flux error:", diagnostics["equilibrated_flux_l2"])
assert diagnostics["fine_cell_balance_defect"] < 1e-10

## 5. Refine the actual pressure and reconstructed flux

Use four uniform macro meshes with 4, 8, 16 and 32 subdivisions per coordinate. Every macrocell keeps the same four-cell local refinement, local P3 degree and P0 trace. Therefore $h=H/2$ and the first-order trace error of the primal method remains present. Lowest-order RT0 additionally has first-order approximation in the physical vector norm for smooth fields.

The plotted order-one lines are **approximation controls for this distinct reconstruction**, not an attribution of the RT2 reconstruction theorem. We report all levels, every consecutive order and the normalized final four-level amplitude. No algebraic tolerance is changed to obtain those rates. The analytical norm and exact fine-cell balance are independent checks.

Set `PYMHM_NOTEBOOK_STUDY=1` before starting the kernel to reacquire all four levels; the normal view reads attributed native records and still executes the explicit eight-subdivision workflow above.

In [ ]:
notebook_path = ROOT/"notebooks/darcy/equilibrated_flux_workflow.ipynb"
source_paths = [notebook_path, ROOT/"examples/tutorial_equilibrated_flux.py",
                ROOT/"examples/tutorial_recovery_acquisition.py", ROOT/"pixi.lock"]
if RUN_STUDY:
    executed_sources = recovery_provenance(ROOT, source_paths)
    rows = []
    for n in (4, 8, 16, 32):
        with threadpool_limits(1):
            solution = solve_declared(TriangleMesh.unit_square(n))
            row, recovered = equilibrated_diagnostics(solution, exact_pressure, exact_gradient)
        row["macro_resolution"] = n
        rows.append(row)
        print(n, row["raw_flux_l2"], row["equilibrated_flux_l2"], row["fine_cell_balance_defect"])
        if n == 16:
            display_samples = equilibrated_display_samples(solution, recovered, exact_pressure, exact_gradient)
    write_recovery_record(
        RECORD, root=ROOT, source_paths=source_paths, source_manifest=executed_sources,
        data={"case": "sin(2*pi*x)sin(2*pi*y), identity diffusion, zero Dirichlet",
              "scope": "Distinct RT0 constrained minimum-distance reconstruction",
              "local_subdivisions": 2, "assembly_quadrature_degree": 20,
              "uniform": rows, "samples": display_samples},
    )
else:
    data = read_recovery_record(RECORD, notebook=notebook_path)
    rows, display_samples = data["uniform"], data["samples"]

series = refinement_series(
    RECORD, rows, "macro_diameter",
    {"Raw physical flux": "raw_flux_l2", "Equilibrated RT0 flux": "equilibrated_flux_l2"},
    {"Raw physical flux": 1.0, "Equilibrated RT0 flux": 1.0},
    method="RT0 constrained equilibration", spaces="P3 pressure, P0 trace, four fine cells per macrocell",
    rate_provenance="Primal trace order and lowest-order RT approximation control; distinct operator",
    root=ROOT,
)
print(asymptotic_summary(series))
plot_method_series(series);

## 6. Inspect reconstructed fields on the refined mesh

The field panels use the actual 16-subdivision macro mesh and its fine cells. Each fine triangle supplies its own centroid value, so interface values are not smoothed. Black lines highlight the macro mesh on analytical, numerical and error panels. Plot samples illustrate the geometry; the norms above integrate the full fields independently.

The higher-order canonical reconstruction and the strict RT0 equilibration answer different questions. Use canonical moments for its stated normal-moment and projected-divergence estimates; use this constrained RT0 reconstruction when exact fine-cell source balance is the requested invariant.

In [ ]:
plot_recovery_fields(display_samples, reconstruction_name="RT0");

## References

- G. R. Barrenechea, L. Martins, W. Pereira and F. Valentin, *An H(div; Ω)-Conforming Flux Reconstruction for the Multiscale Hybrid-Mixed Method*, Multiscale Modeling & Simulation 24(2), 399–428 (2026), [doi:10.1137/24M1673073](https://doi.org/10.1137/24M1673073). The smooth test data and primal-method estimates are used here; the distinct constrained RT0 operator is explicitly identified above.
- C. Harder, D. Paredes and F. Valentin, *A family of multiscale hybrid-mixed finite element methods for the Darcy equation with rough coefficients*, Journal of Computational Physics 245, 107–130 (2013), [doi:10.1016/j.jcp.2013.03.019](https://doi.org/10.1016/j.jcp.2013.03.019).

Figures are licensed under CC BY 4.0. Attribute IPES Research Group when reusing or adapting them.